# Return Explorer Dashboard

Module: Markets and Data

## Lesson summary

This dashboard lets students explore prices, returns, rolling volatility, drawdowns, return distributions, and basic risk/performance metrics from the same clean price panel. It uses reusable data helpers instead of embedding provider-specific logic in the notebook.

The dashboard is exploratory, not a recommendation engine. It should help the reader compare assets, identify risk patterns, and decide what deserves deeper analysis {cite}`few2006dashboard,cairo2016truthful`.

## Learning objectives

By the end of this dashboard, students should be able to:

- convert prices into simple or log returns;
- inspect cumulative performance and drawdown;
- compare rolling volatility across assets;
- identify skewness and tail behavior from histograms;
- read introductory metrics such as downside percentiles, historical VaR as a percentile, Sharpe ratio, hit ratio, and benchmark correlation;
- connect dashboard views to a short written insight;
- use the same return matrix as an input to risk and portfolio modules.

## Setup

In [ ]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from ipywidgets import Dropdown, IntSlider, interact
from plotly.subplots import make_subplots

from src.market_data import DEFAULT_RETURN_DASHBOARD_TICKERS, return_dashboard_price_panel, returns_from_prices

DATA_MODE = os.getenv("DATA_MODE", "offline").lower()
LIVE_DATA_START = os.getenv("LIVE_DATA_START", "2020-01-01")
LIVE_DATA_END = os.getenv("LIVE_DATA_END", "2024-12-31")
RUN_INTERACTIVE_WIDGETS = os.getenv("RUN_INTERACTIVE_WIDGETS", "1") == "1"

## Price and return panel

In [ ]:
prices = return_dashboard_price_panel(
    data_mode=DATA_MODE,
    start=LIVE_DATA_START,
    end=LIVE_DATA_END,
)
prices.tail()

In [ ]:
pd.DataFrame(
    [
        {
            "data_mode": prices.attrs.get("data_mode", DATA_MODE),
            "sources": prices.attrs.get("sources", "not specified"),
            "tickers": prices.attrs.get("tickers", DEFAULT_RETURN_DASHBOARD_TICKERS),
            "start": prices.index.min(),
            "end": prices.index.max(),
            "observations": len(prices),
        }
    ]
)

## Dashboard question map

Each view should answer a specific question.

| Question | Dashboard element |
| --- | --- |
| How did the asset evolve over time? | Price chart |
| How did cumulative performance evolve? | Cumulative return chart |
| How dispersed are returns? | Return histogram |
| How did risk change over time? | Rolling volatility chart |
| How severe were losses from peaks? | Drawdown line |
| How does the asset compare with a benchmark? | Risk summary and benchmark correlation |
| Can another analyst trust the panel? | Data mode, source, date range, and assumptions |

Chart choice should support the analytical question. Visual design should prioritize clarity, labels, units, comparable scales, and truthful representation {cite}`tufte2001visual,wilke2019dataviz`.

## Exploratory risk and performance metrics

Return alone is not enough. At the exploratory level, risk can be summarized through volatility, downside returns, drawdowns, tail percentiles, simple historical VaR, return-to-risk ratios, and stability over time.

This table is a first reading, not a complete risk model. Drawdown is useful because investors often experience risk through losses from previous highs, while historical VaR is only a distributional threshold, not a worst-case loss {cite}`magdonismail2004drawdown,jorion2007var`. Full VaR modeling, expected shortfall, stress testing, backtesting, volatility modeling, and attribution belong in later modules.

In [ ]:
dashboard_returns = returns_from_prices(prices, method="log")


def drawdown(cumulative_returns):
    wealth_index = 1 + cumulative_returns
    running_peak = wealth_index.cummax()
    return wealth_index / running_peak - 1


def summarize_return_series(asset_returns, benchmark_returns=None):
    asset_returns = asset_returns.dropna()
    cumulative_return = np.exp(asset_returns.cumsum()) - 1
    asset_drawdown = drawdown(cumulative_return)
    annualized_return = asset_returns.mean() * 252
    annualized_volatility = asset_returns.std() * np.sqrt(252)
    sharpe_ratio = annualized_return / annualized_volatility if annualized_volatility else np.nan
    benchmark_correlation = (
        asset_returns.corr(benchmark_returns.dropna())
        if benchmark_returns is not None
        else np.nan
    )
    return pd.Series(
        {
            "total_return": cumulative_return.iloc[-1],
            "annualized_return": annualized_return,
            "annualized_volatility": annualized_volatility,
            "best_return": asset_returns.max(),
            "worst_return": asset_returns.min(),
            "p05_return": asset_returns.quantile(0.05),
            "historical_var_5pct": -asset_returns.quantile(0.05),
            "maximum_drawdown": asset_drawdown.min(),
            "sharpe_ratio_no_rf": sharpe_ratio,
            "hit_ratio": (asset_returns > 0).mean(),
            "correlation_with_benchmark": benchmark_correlation,
        }
    )


BENCHMARK_ASSET = prices.columns[0]
risk_summary = pd.DataFrame(
    {
        asset: summarize_return_series(
            dashboard_returns[asset],
            benchmark_returns=dashboard_returns[BENCHMARK_ASSET],
        )
        for asset in prices.columns
    }
).T
risk_summary

The Sharpe ratio shown here uses a zero risk-free rate simplification. In applied work, the reference rate should match the currency, horizon, and context of the analysis {cite}`sharpe1994ratio,lo2002sharpe`.

## Dashboard function

In [ ]:
def plot_return_explorer(asset=None, return_method="log", rolling_window=63):
    if asset is None:
        asset = prices.columns[0]

    returns = returns_from_prices(prices, method=return_method)
    asset_returns = returns[asset].dropna()
    if return_method == "log":
        cumulative = np.exp(asset_returns.cumsum()) - 1
    else:
        cumulative = (1 + asset_returns).cumprod() - 1
    rolling_volatility = asset_returns.rolling(rolling_window).std() * np.sqrt(252)
    asset_drawdown = drawdown(cumulative)

    fig = make_subplots(
        rows=2,
        cols=2,
        shared_xaxes=False,
        vertical_spacing=0.14,
        subplot_titles=("Price", "Cumulative return", "Rolling volatility", "Return histogram"),
    )
    fig.add_trace(go.Scatter(x=prices.index, y=prices[asset], mode="lines", name="price"), row=1, col=1)
    fig.add_trace(
        go.Scatter(x=cumulative.index, y=cumulative, mode="lines", name="cumulative return"),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter(x=asset_drawdown.index, y=asset_drawdown, mode="lines", name="drawdown"),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter(x=rolling_volatility.index, y=rolling_volatility, mode="lines", name="rolling vol"),
        row=2,
        col=1,
    )
    fig.add_trace(go.Histogram(x=asset_returns, nbinsx=50, name="returns"), row=2, col=2)
    fig.update_layout(
        title=f"Return explorer: {asset} ({prices.attrs.get('data_mode', DATA_MODE)} data)",
        template="plotly_white",
        height=720,
        showlegend=False,
    )
    fig.update_yaxes(tickformat=".1%", row=1, col=2)
    fig.update_yaxes(tickformat=".1%", row=2, col=1)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


DEFAULT_ASSET = "mexican_equity" if "mexican_equity" in prices.columns else prices.columns[0]

if RUN_INTERACTIVE_WIDGETS:
    interact(
        plot_return_explorer,
        asset=Dropdown(options=list(prices.columns), value=DEFAULT_ASSET),
        return_method=Dropdown(options=["log", "simple"], value="log"),
        rolling_window=IntSlider(value=63, min=21, max=252, step=21),
    );
else:
    plot_return_explorer()

## Interpretation checklist

Avoid these common mistakes:

| Mistake | Why it is a problem |
| --- | --- |
| Ranking assets only by average return | Ignores risk and drawdowns |
| Treating volatility as complete risk | Ignores asymmetry and tail losses |
| Ignoring the sample period | Results may be regime-dependent |
| Annualizing mechanically | Square-root scaling assumptions may not hold |
| Comparing assets in different currencies | Mixes asset return and FX return |
| Using price return when total return is needed | Ignores dividends or distributions |
| Treating VaR as maximum possible loss | VaR is a threshold, not a worst-case loss |
| Treating Sharpe ratio as absolute truth | It depends on assumptions and return behavior |

## Real-data extension

The default build uses `DATA_MODE=offline`. To run the same dashboard with public market prices, launch Jupyter locally with live mode:

```bash
DATA_MODE=live LIVE_DATA_START=2020-01-01 LIVE_DATA_END=2024-12-31 uv run jupyter lab
```

The live panel is loaded through `return_dashboard_price_panel`, which maps public tickers to stable dashboard labels and caches provider extracts before analysis.